# Who are the 152 hedge funds?

Classifies the Cayman entities in `hf_Valeri.xlsx` by strategy, using only public sources:
**GLEIF** (names + fund manager), **SEC Form ADV** (adviser per fund, matched on LEI), and
**Part 2A brochures** (strategy text). Needs a normal internet connection.

Run top to bottom. Everything downloaded is cached in `fund_classification/`, so re-running is cheap.
One edit below: put your email in `EMAIL` (the SEC requires an identifying User-Agent).

In [ ]:
import json, re, time, zipfile, difflib
from pathlib import Path

import pandas as pd
import requests

EMAIL = "your.name@example.com"          # EDIT ME (required by the SEC)
UA = {"User-Agent": f"JMP fund identification research {EMAIL}"}

ROOT = Path.cwd() if (Path.cwd() / "hf_Valeri.xlsx").exists() else Path.cwd().parent
OUT = ROOT / "fund_classification"
for sub in ["cache", "sec", "brochures"]:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

funds = pd.read_excel(ROOT / "hf_Valeri.xlsx").rename(columns={"entity_id": "lei"})
funds["lei"] = funds["lei"].str.strip().str.upper()
print(len(funds), "funds,", funds["name"].isna().sum(), "without a name")

## 1. GLEIF: fill missing names, get the fund manager (~3 min, cached)

In [ ]:
def gleif(url, key):
    f = OUT / "cache" / f"{key}.json"
    if f.exists():
        return json.loads(f.read_text())
    r = requests.get(url, timeout=30)
    time.sleep(1.05)                                   # GLEIF rate limit ~1/sec
    data = r.json() if r.ok else {}
    f.write_text(json.dumps(data))
    return data

rows = []
for i, lei in enumerate(funds["lei"], 1):
    rec = gleif(f"https://api.gleif.org/api/v1/lei-records/{lei}", lei)
    ent = rec.get("data", {}).get("attributes", {}).get("entity", {})
    row = {"lei": lei,
           "gleif_name": (ent.get("legalName") or {}).get("name"),
           "gleif_status": ent.get("status"),
           "manager_name": None}
    rel = rec.get("data", {}).get("relationships", {})
    link = next((v["links"]["related"] for k, v in rel.items()
                 if "fund-manager" in k and v.get("links", {}).get("related")), None)
    if link:
        m = gleif(link, f"{lei}_mgr").get("data", {})
        if isinstance(m, dict) and m.get("id"):
            row["manager_name"] = (m["attributes"]["entity"].get("legalName") or {}).get("name")
    rows.append(row)
    if i % 25 == 0:
        print(i, "/", len(funds))

funds = funds.merge(pd.DataFrame(rows), on="lei", how="left")
funds["name"] = funds["name"].fillna(funds["gleif_name"])
print("managers found:", funds["manager_name"].notna().sum())
print(funds["manager_name"].value_counts().head(15))

## 2. Classify by name

Entity names often carry the strategy ("Millennium **Fixed Income**", "Garda **FIRV**") — this
classifies the sleeve, the right unit for the repo data. Manager names get the same rules.

In [ ]:
RULES = [
    ("Fixed income / rates RV", ["FIXED INCOME", "FIRV", "RELATIVE VALUE", " RATES", "G-10",
                                 "GLOBAL RATES", "INFLATION", "BOND", "TERM CREDIT", "CONVEX",
                                 "TAIL RISK", "VOLATILITY"]),
    ("Global macro",  ["MACRO", "ALL WEATHER", "PURE ALPHA", "OPTIMAL PORTFOLIO", "DMO"]),
    ("Credit",        ["CREDIT", "ABS ", "HIGH YIELD", "DISTRESSED"]),
    ("Equity",        ["EQUITY"]),
    ("Commodity",     ["COMMODITY"]),
    ("Multi-strategy platform", ["MULTI-STRATEGY", "MULTI STRATEGY", "DIVERSIFIED ALPHA"]),
]

def classify(text):
    if not isinstance(text, str):
        return None
    t = " " + re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9\- ]", " ", text.upper())) + " "
    return next((lab for lab, kws in RULES if any(k in t for k in kws)), None)

funds["class_entity_name"] = funds["name"].map(classify)
funds["class_manager_name"] = funds["manager_name"].map(classify)
print(funds["class_entity_name"].value_counts(dropna=False))

## 3. SEC Form ADV: which adviser runs each fund?

Downloads the two current bulk archives (registered + exempt advisers, ~100–200 MB total) and matches
your funds' LEIs in the Schedule D 7.B.(1) private-fund file. If the download cell fails, grab the
two newest zips by hand from the
[SEC page](https://www.sec.gov/data-research/sec-markets-data/information-about-registered-investment-advisers-exempt-reporting-advisers)
into `fund_classification/sec/` and re-run from the cell after it.

In [ ]:
SEC_PAGE = ("https://www.sec.gov/data-research/sec-markets-data/"
            "information-about-registered-investment-advisers-exempt-reporting-advisers")

zips = sorted((OUT / "sec").glob("*.zip"))
if not zips:
    html = requests.get(SEC_PAGE, headers=UA, timeout=60).text
    links = ["https://www.sec.gov" + l if l.startswith("/") else l
             for l in re.findall(r'href="([^"]+\.zip)"', html)]
    for pat in ["ia", "era"]:
        cand = sorted(l for l in links if pat in Path(l).name.lower())
        if cand:
            url = cand[-1]
            print("downloading", url)
            (OUT / "sec" / Path(url).name).write_bytes(requests.get(url, headers=UA, timeout=600).content)
    zips = sorted((OUT / "sec").glob("*.zip"))
print("archives:", [z.name for z in zips])

In [ ]:
def read_csvs(zpath, name_contains):
    frames = []
    with zipfile.ZipFile(zpath) as z:
        for n in z.namelist():
            if n.lower().endswith(".csv") and name_contains.lower() in Path(n).name.lower():
                frames.append(pd.read_csv(z.open(n), dtype=str, encoding="latin-1",
                                          low_memory=False, on_bad_lines="skip"))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

pf = pd.concat([read_csvs(z, "7B1") for z in zips], ignore_index=True)     # private funds
base = pd.concat([read_csvs(z, "Base") for z in zips], ignore_index=True)  # advisers
print("private-fund rows:", len(pf), "| adviser rows:", len(base))

# find the relevant columns by content / name
LEI_RE = re.compile(r"^[A-Z0-9]{18}[0-9]{2}$")
def detect(df, test):
    shares = {c: df[c].dropna().astype(str).str.strip().str.upper().head(2000)
                    .map(lambda x: bool(test(x))).mean() for c in df.columns}
    return max(shares, key=shares.get)

lei_col = detect(pf, LEI_RE.match)
fname_col = detect(pf, lambda x: "FUND" in x or "MASTER" in x or "LTD" in x)
crd_pf = next(c for c in pf.columns if "CRD" in c.upper())
crd_base = next(c for c in base.columns if "CRD" in c.upper())
aname = next(c for c in base.columns if "LEGAL" in c.upper() or "NAME" in c.upper())
print("columns -> LEI:", lei_col, "| fund name:", fname_col, "| CRD:", crd_pf, "/", crd_base, "| adviser:", aname)

In [ ]:
pf["_lei"] = pf[lei_col].astype(str).str.strip().str.upper()
m = funds.merge(pf.drop_duplicates("_lei"), left_on="lei", right_on="_lei", how="left")
print("LEI matches:", m["_lei"].notna().sum(), "/", len(funds))

# fallback: normalized-name match for the unmatched
def norm(s):
    s = re.sub(r"[^A-Z0-9 ]", " ", str(s).upper())
    s = re.sub(r"\b(THE|LTD|LIMITED|LP|L P|LLC|LDC|INC|FUND|MASTER|CAYMAN)\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()

pf["_norm"] = pf[fname_col].map(norm)
lookup = pf.drop_duplicates("_norm").set_index("_norm")
for i in m.index[m["_lei"].isna() & m["name"].notna()]:
    n = norm(m.at[i, "name"])
    hit = n if n in lookup.index else next(iter(difflib.get_close_matches(n, lookup.index, 1, 0.92)), None)
    if hit:
        m.loc[i, pf.columns] = lookup.loc[hit].reindex(pf.columns).values
m["matched"] = m["_lei"].notna() | m[fname_col].notna()
print("total matches:", m["matched"].sum(), "/", len(funds))

base["_crd"] = base[crd_base].astype(str).str.strip()
m["_crd"] = m[crd_pf].astype(str).str.strip()
m = m.merge(base.drop_duplicates("_crd")[["_crd", aname]].rename(columns={aname: "adviser_name"}),
            on="_crd", how="left")
m["iapd_link"] = "https://adviserinfo.sec.gov/firm/summary/" + m["_crd"]

advisers = (m.dropna(subset=["adviser_name"]).groupby(["adviser_name", "_crd"])
              .size().rename("n_funds").reset_index().sort_values("n_funds", ascending=False))
print(advisers.head(25).to_string(index=False))

## 4. Brochures: adviser strategy text

Tries to auto-download each matched adviser's Part 2A brochure; where that fails it prints the IAPD
link — open it, "Part 2 Brochures" tab, save the PDF as `fund_classification/brochures/<CRD>.pdf`.
The classifier reads whatever PDFs are in that folder.

In [ ]:
def fetch_brochure(crd):
    dest = OUT / "brochures" / f"{crd}.pdf"
    if dest.exists():
        return "have"
    try:
        j = requests.get(f"https://api.adviserinfo.sec.gov/search/brochure/{crd}",
                         headers=UA, timeout=30).text
        vid = re.search(r'"(?:brchrVrsnID|versionId)"\s*:\s*"?(\d+)', j)
        if vid:
            pdf = requests.get("https://files.adviserinfo.sec.gov/IAPD/Content/Common/"
                               f"crd_iapd_Brochure.aspx?BRCHR_VRSN_ID={vid.group(1)}",
                               headers=UA, timeout=60)
            if pdf.ok and pdf.content[:4] == b"%PDF":
                dest.write_bytes(pdf.content)
                return "downloaded"
    except Exception:
        pass
    return "manual"

for _, r in advisers.iterrows():
    s = fetch_brochure(r["_crd"])
    if s == "manual":
        print(f"manual: {r['adviser_name']}  ->  https://adviserinfo.sec.gov/firm/summary/{r['_crd']}")
    time.sleep(0.5)
print("brochures on disk:", len(list((OUT / "brochures").glob("*.pdf"))))

In [ ]:
from pypdf import PdfReader     # pip install pypdf

BRO_RULES = [
    ("Fixed income / rates RV", ["fixed income", "relative value", "sovereign", "government bond",
                                 "interest rate", "repurchase agreement", "bond futures"]),
    ("Global macro",  ["global macro", "macroeconomic"]),
    ("Credit",        ["credit", "high yield", "distressed"]),
    ("Equity",        ["equity", "equities", "stock selection"]),
    ("Multi-strategy platform", ["multi-strategy", "multiple strategies"]),
]

def classify_pdf(p):
    try:
        text = " ".join((pg.extract_text() or "") for pg in PdfReader(str(p)).pages).lower()
    except Exception:
        return None
    hits = {lab: sum(text.count(k) for k in kws) for lab, kws in BRO_RULES}
    best = max(hits, key=hits.get)
    return best if hits[best] >= 3 else None

bro = {p.stem: classify_pdf(p) for p in (OUT / "brochures").glob("*.pdf")}
m["class_brochure"] = m["_crd"].map(bro)
print(m["class_brochure"].value_counts(dropna=False))

## 5. Final table

In [ ]:
m["class_final"] = (m["class_entity_name"].fillna(m["class_brochure"])
                    .fillna(m["class_manager_name"]).fillna("Manual review"))

profiles = m[["lei", "name", "manager_name", "adviser_name", "_crd", "iapd_link",
              "class_entity_name", "class_brochure", "class_manager_name", "class_final"]]
profiles.to_csv(OUT / "fund_profiles.csv", index=False)

summary = profiles["class_final"].value_counts().rename_axis("strategy").rename("n_funds").reset_index()
summary["share_pct"] = (100 * summary["n_funds"] / len(profiles)).round(1)
summary.to_csv(OUT / "summary.csv", index=False)
print(summary.to_string(index=False), "\n")

for _, r in summary.iterrows():                       # LaTeX rows for the paper
    print(f"{r['strategy']} & {r['n_funds']} & {r['share_pct']} \\\\")

print("\n--- manual review ---")
print(profiles.loc[profiles["class_final"] == "Manual review",
                   ["name", "manager_name", "adviser_name", "iapd_link"]].to_string(index=False))

**Volume weighting (ECB-side):** merge `fund_profiles.csv` with the internal per-fund volumes
(`hf.xlsx` from `build_main_panel.ipynb`) on `lei` = `entity_id` and aggregate volume by `class_final`.
Only the aggregated shares go into the paper.